# Baseline workflow
As a reference experiment, a **baseline training setup without any data augmentation** was evaluated.  
This configuration serves as a lower-bound performance benchmark and enables direct comparison with augmentation-based training strategies.

---

### Dataset
Training and validation data were drawn from multiple preprocessed image sources:

- processed_tseg_jpgs  
- processed_lsd_jpgs  
- processed_spider_jpgs  
- processed_osf_jpgs  

All images were resized to a fixed resolution and used **without any geometric or intensity transformations**.

---

### Models Evaluated
The following architectures were trained under identical baseline conditions:

- **U-Net**
- **Res-U-Net**
- **Attention Res-U-Net**

All models predicted multi-channel vertebra heatmaps from single-channel grayscale MRI inputs.

---

### Training Configuration
- Loss function: **Weighted Mean Squared Error (Weighted MSE)**
- Optimizer: **Adam**
- Learning rate: fixed
- Heatmap sigma: fixed (no curriculum)
- Data augmentation: **none**
- Resolution: fixed
- Weight decay: disabled

---

### Observations
- **Res-U-Net** consistently outperformed U-Net and Attention Res-U-Net, even in the absence of data augmentation.
- **Attention Res-U-Net** did not provide performance gains, likely due to increased architectural complexity relative to dataset size.

---

### Notable Finding: Weight Decay Instability
Enabling **weight decay (L2 regularization)** caused **training divergence** across all tested architectures.
As a result, weight decay was disabled in all subsequent experiments.

## Data preprocessing

In [1]:
import os, glob
import numpy as np
import pandas as pd
import zipfile
from pathlib import Path
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image
import cv2
import re
from PIL import Image
import torch
from torch.utils.data import Dataset
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from tqdm import tqdm
from torch.utils.data import random_split, DataLoader
import math

2026-04-29 11:36:39.806640: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-29 11:36:39.819645: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777455399.832692 1750283 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777455399.836624 1750283 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-29 11:36:39.852140: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
%run ./Pretraining-fce.ipynb

In [3]:
%run ./Pretraining-models.ipynb

In [4]:
%run ./Pretraining-datasets.ipynb

In [5]:
BASE_DIR = Path.cwd()
print(BASE_DIR)

/home/jupyter-lukj08@vse.cz/VSE_bachelor_thesis_lumbar_spine_degeneration_classification/Localization-pretrain


In [6]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"       # quieter logs (optional)
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"  # enables growth mode
torch.cuda.is_available(), torch.cuda.get_device_name(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [8]:
df_coords = pd.read_csv("/home/jupyter-lukj08@vse.cz/VSE_bachelor_thesis_lumbar_spine_degeneration_classification/BC-data/data-pretraining/df_coords.csv")
print("Loaded df_coords:")
display(df_coords)

Loaded df_coords:


,filename,source,x,y,level,relative_x,relative_y,full_path,exists
0,1_t2.jpg,spider,139,175,L5/S1,0.542969,0.683594,./data/processed_spider_jpgs/1_t2.jpg,True
1,1_t2.jpg,spider,133,157,L4/L5,0.519531,0.613281,./data/processed_spider_jpgs/1_t2.jpg,True
2,1_t2.jpg,spider,132,131,L3/L4,0.515625,0.511719,./data/processed_spider_jpgs/1_t2.jpg,True
3,1_t2.jpg,spider,131,102,L2/L3,0.511719,0.398438,./data/processed_spider_jpgs/1_t2.jpg,True
4,1_t2.jpg,spider,134,84,L1/L2,0.523438,0.328125,./data/processed_spider_jpgs/1_t2.jpg,True
...,...,...,...,...,...,...,...,...,...
5600,ID39.jpg,osf,138,198,L5/S1,0.539062,0.773438,./data/processed_osf_jpgs/ID39.jpg,True
5601,ID39.jpg,osf,132,181,L4/L5,0.515625,0.707031,./data/processed_osf_jpgs/ID39.jpg,True
5602,ID39.jpg,osf,127,163,L3/L4,0.496094,0.636719,./data/processed_osf_jpgs/ID39.jpg,True
5603,ID39.jpg,osf,126,140,L2/L3,0.492188,0.546875,./data/processed_osf_jpgs/ID39.jpg,True


In [10]:
path = "/home/jupyter-lukj08@vse.cz/VSE_bachelor_thesis_lumbar_spine_degeneration_classification/BC-data/data-pretraining/processed_osf_jpgs/ID39.jpg"
img = Image.open(path)
print(img.size) 

(256, 256)


In [11]:
# Normalize for training
IMG_SIZE = 256
mean = [0.2127]
std = [0.2673]

img_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

ds = SpineLandmarksDataset(
    df_coords,
    img_size=IMG_SIZE,
    sigma=5,
    transform=img_transform,
)

print("Number of images:", len(ds))
print("Levels / channels:", ds.levels)

img, target = ds[0]
print("img.shape    =", img.shape)
print("target.shape =", target.shape)

Number of images: 1121
Levels / channels: ['L1_L2', 'L2_L3', 'L3_L4', 'L4_L5', 'L5_S1']


FileNotFoundError: [Errno 2] No such file or directory: './data/processed_spider_jpgs/1_t2.jpg'

In [ ]:
image_classes = [
    'processed_tseg_jpgs',
    'processed_lsd_jpgs',
    'processed_spider_jpgs',
    'processed_osf_jpgs'
]
destination_dir = Path("./data")

plot_images_with_landmarks(image_classes, df_coords)

## Dataset preparation

In [ ]:
# ---- Split (no leakage) ----
df_train, df_val, df_test = split_by_scan(df_coords, val_frac=0.2, seed=42)

In [ ]:
train_ds = SpineLandmarksDataset(
    df_train,
    img_size=IMG_SIZE,
    sigma=5.0,
    transform=img_transform
)

val_ds = SpineLandmarksDataset(
    df_val,
    img_size=IMG_SIZE,
    sigma=5.0,
    transform=img_transform,

)

test_ds = SpineLandmarksDatasetStaticAug(
    df_test,
    img_size=IMG_SIZE,
    sigma=5.0,
    transform=img_transform,

)

print(f"Train samples: {len(train_ds)}, Val samples: {len(val_ds)}, Teat samples: {len(test_ds)}")

In [ ]:
# DataLoaders
BATCH_SIZE = 8
NUM_WORKERS = 2

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=False
)

val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=False
)

test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=False
)

print("Train batches:", len(train_loader), "Val batches:", len(val_loader))
print("Levels:", train_ds.levels)

## Models training

In [ ]:
# CONFIG: 
criterion = nn.MSELoss()
num_epochs = 30          # you can change this
patience   = 5           # early stopping patience

# Automatic Mixed Precision configuration
use_amp = torch.cuda.is_available()  # True only if GPU is available
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


best_val_loss = math.inf
best_state = None
epochs_without_improve = 0

# Automatic Mixed Precision configuration
use_amp = torch.cuda.is_available()  # True only if GPU is available
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

### U-Net

In [ ]:
model1 = UNet(
    in_channels=1,   # grayscale
    n_classes=K,
).to(device)

optimizer1 = torch.optim.Adam(model1.parameters(), lr=1e-3)

print("Model ready on", device)

total_params = sum(p.numel() for p in model1.parameters())
print(f"Total parameters: {total_params:,}")

In [ ]:
model1, history1 = train_basic(
    model=model1,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer1,
    device=device,
    criterion=criterion,
    num_epochs=num_epochs,
    patience=patience,
)


In [ ]:
model2 = ResUNet(
    in_channels=1,   # grayscale
    n_classes=K,
    base_c=64        # matches the diagram; you can use 32 if you want smaller model
).to(device)

optimizer2 = torch.optim.Adam(model2.parameters(), lr=1e-3) #weight decay 

print("Model ready on", device)

total_params = sum(p.numel() for p in model2.parameters())
print(f"Total parameters: {total_params:,}")

In [ ]:
model2, history2 = train_basic(
    model=model2,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer2,
    device=device,
    criterion=criterion,
    num_epochs=num_epochs,
    patience=patience,
)


### Attention Res-UNet

In [ ]:
model3 = AttentionResUNet(
    in_channels=1,   # grayscale
    n_classes=5,
    base_c=64  
).to(device)

optimizer3 = torch.optim.Adam(model3.parameters(), lr=1e-3, weight_decay=1e-4)

print("Model ready on", device)

total_params = sum(p.numel() for p in model3.parameters())
print(f"Total parameters: {total_params:,}")

In [ ]:
model3, history3 = train_basic(
    model=model3,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer3,
    device=device,
    criterion=criterion,
    num_epochs=num_epochs,
    patience=patience,
)


## Models Evaluation
The primary evaluation metric is **pixel localization error**:

- Euclidean distance (in pixels) between:
  - Ground truth heatmap argmax
  - Predicted heatmap argmax

Reported metrics include:
- Per-level mean pixel error
- Overall mean pixel error
- Overall 90th percentile (p90) pixel error

---
- In the absence of data augmentation, **simpler architectures generalize better**
- **Attention mechanisms are harmful** in this setting without strong regularization or large datasets

In [ ]:
# Validation
table_val, rows_val = compare_models_pixel_error_heatmap(
    models=[model1, model2, model3],
    model_names=["UNet", "Res-UNet", "Attention-Res-UNet"],
    loader=val_loader,
    device=device,
    use_amp=True,
)

In [ ]:
# Test
table_test, rows_test = compare_models_pixel_error_heatmap(
    models=[model1, model2, model3],
    model_names=["UNet", "Res-UNet", "Attention-Res-UNet"],
    loader=test_loader,
    device=device,
    use_amp=True,
)

### Best models performance visualization

In [ ]:
mean_t = torch.tensor(mean, dtype=torch.float32)
std_t  = torch.tensor(std,  dtype=torch.float32)

for idx in range(10):
    print(f"\nVisualizing val sample {idx}/{len(val_ds)-1}")

    visualize_sample(
        model=model2,
        dataset= test_ds,
        idx=idx,
        device=device,
        mean=mean_t,
        std=std_t,
    )